In [9]:
#Task 14: LangGraph
#Implementing LangGraph React agent

from dataclasses import dataclass
from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama
import re

# -----------------------------
#  Define the state
# -----------------------------
@dataclass
class State:
    question: str
    llm_reply: str = ""
    result: float = 0.0
    answer: str = ""

# -----------------------------
#  Define tools for each operation
# -----------------------------
def add(a, b):
    return {"result": a + b}

def subtract(a, b):
    return {"result": a - b}

def multiply(a, b):
    return {"result": a * b}

def divide(a, b):
    if b == 0:
        return {"result": "Error: division by zero"}
    return {"result": a / b}

# Map LLM Action strings to tools
TOOLS = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide,
}

# -----------------------------
#  LLM node: ask what to do
# -----------------------------
llm = ChatOllama(model="llama3", temperature=0)

def ask_llm(state: State) -> dict:
    prompt = f"""
You are a math agent. 

If an operation is needed, respond EXACTLY like this:

Action: <operation_name>
Action Input: <number1>,<number2>

Question: {state.question}
"""
    response = llm.invoke(prompt)
    return {"llm_reply": response.content.strip()}

# -----------------------------
# parse LLM reply and call the right tool
# -----------------------------
def perform_operation(state: State) -> dict:
    reply = state.llm_reply
    # Extract action
    action_match = re.search(r"Action:\s*(\w+)", reply, re.IGNORECASE)
    numbers_match = re.findall(r"-?\d+\.?\d*", reply)  # handle floats & negatives
    
    if not action_match or len(numbers_match) != 2:
        return {"result": "Error: could not parse action or numbers"}
    
    action = action_match.group(1).lower()
    a, b = float(numbers_match[0]), float(numbers_match[1])
    
    if action in TOOLS:
        tool_out = TOOLS[action](a, b)
        return {"result": tool_out["result"]}
    
    return {"result": f"Error: unknown action '{action}'"}

# -----------------------------
#  Node: generate final answer
# -----------------------------
def final_answer(state: State) -> dict:
    return {"answer": f"The answer is {state.result}"}

# -----------------------------
# Build the LangGraph
# -----------------------------
graph = StateGraph(State)
graph.add_node("ask_llm", ask_llm)
graph.add_node("perform_operation", perform_operation)
graph.add_node("final_answer", final_answer)

graph.add_edge(START, "ask_llm")
graph.add_edge("ask_llm", "perform_operation")
graph.add_edge("perform_operation", "final_answer")
graph.add_edge("final_answer", END)

compiled = graph.compile()

# -----------------------------
# Run examples
# -----------------------------
examples = [
    "What is 7 multiplied by 6?",
    "Add 12 and 8",
    "Divide 42 by 7",
    "Subtract 10 from 25",
]

for q in examples:
    state = State(question=q)
    result = compiled.invoke(state)
    print(f"Q: {q} → {result['answer']}")

Q: What is 7 multiplied by 6? → The answer is 42.0
Q: Add 12 and 8 → The answer is Error: unknown action 'addition'
Q: Divide 42 by 7 → The answer is 6.0
Q: Subtract 10 from 25 → The answer is 15.0
